In [64]:
import hashlib
import json
import time
import os

def sha256(data: str) -> str:
    return hashlib.sha256(data.encode("utf-8")).hexdigest()

In [85]:
class Block:
    def __init__(self, index: int, timestamp: float, data: dict, previous_hash: str, nonce: int = 0):
        self.index = index
        self.timestamp = timestamp
        self.data = data  # This will now contain file content and filename
        self.previous_hash = previous_hash
        self.nonce = nonce
        self.hash = self.compute_hash()

    def compute_hash(self) -> str:
        # Deterministic serialization ensures if content changes, hash changes
        block_string = json.dumps({
            "index": self.index,
            "timestamp": self.timestamp,
            "data": self.data,
            "previous_hash": self.previous_hash,
            "nonce": self.nonce
        }, sort_keys=True, separators=(",", ":"))
        return sha256(block_string)

In [86]:
class Blockchain:
    def __init__(self, difficulty: int = 3):
        self.chain = []
        self.difficulty = difficulty
        self.create_genesis_block()

    def create_genesis_block(self):
        # Genesis block starts the chain with previous_hash "0"
        genesis = Block(index=0, timestamp=time.time(), data={"msg": "genesis"}, previous_hash="0")
        self.mine_block(genesis)
        self.chain.append(genesis)

    @property
    def last_block(self) -> Block:
        return self.chain[-1]

    def is_valid_proof(self, block: Block) -> bool:
        return block.hash.startswith("0" * self.difficulty)

    def mine_block(self, block: Block) -> Block:
        while True:
            block.hash = block.compute_hash()
            if self.is_valid_proof(block):
                return block
            block.nonce += 1

    def add_block(self, file_path: str) -> Block:
        # Read the actual content of the file
        with open(file_path, 'r') as f:
            content = f.read()
        
        # Store both filename and content in the block data
        data = {
            "filename": os.path.basename(file_path),
            "content": content
        }
        
        new_block = Block(
            index=self.last_block.index + 1,
            timestamp=time.time(),
            data=data,
            previous_hash=self.last_block.hash
        )
        self.mine_block(new_block)
        
        # Integrity check
        if new_block.previous_hash != self.last_block.hash or not self.is_valid_proof(new_block):
            raise ValueError("Invalid block—rejected.")
            
        self.chain.append(new_block)
        return new_block

    def is_chain_valid(self) -> bool:
        if len(self.chain) <= 1:
            print("✅ Genesis block is valid.")
            return True

        for i in range(1, len(self.chain)):
            curr = self.chain[i]
            prev = self.chain[i - 1]

            curr_name = curr.data.get("filename", "Unknown")
            prev_name = prev.data.get("filename", "Genesis")

            if curr.previous_hash != prev.hash:
                print(f"❌ LINK BROKEN: [{prev_name} -> {curr_name}]")
                print(f"   Reason: previous_hash in {curr_name} does not match {prev_name}'s hash.")
                return False

        
            if not self.is_valid_proof(curr):
                print(f"❌ SECURITY FAILURE: {curr_name}")
                print(f"   Reason: Hash does not satisfy difficulty (leading zeros).")
                return False

        # If all checks pass for this specific link
            print(f"✅ Link [{prev_name} -> {curr_name}] is VALID.")

        print("\n🏆 FINAL RESULT: Blockchain is fully synchronized and secure.")
        return True

In [87]:
if __name__ == "__main__":
    bc = Blockchain(difficulty=4)
    
    # List of files to add to the blockchain
    files = ["a1.txt", "a2.txt", "a3.txt"]
    
    for file_name in files:
        if os.path.exists(file_name):
            print(f"Adding {file_name} to the blockchain...")
            bc.add_block(file_name)
        else:
            print(f"File {file_name} not found.")

    # Print results
    for b in bc.chain:
        print(f"Index: {b.index} | File: {b.data.get('filename', 'N/A')}")
        print(f"Content: {b.data.get('content', 'N/A')}")
        print(f"Hash: {b.hash}")
        print("-" * 60)

    print("Chain valid?", bc.is_chain_valid())

Adding a1.txt to the blockchain...
Adding a2.txt to the blockchain...
Adding a3.txt to the blockchain...
Index: 0 | File: N/A
Content: N/A
Hash: 000086580120f85c0c7d56f37d7dc970ebabbda5212cc582e7c3d1271986091f
------------------------------------------------------------
Index: 1 | File: a1.txt
Content: hello.
Hash: 0000a90264b313a4155790922976d335c2ef3733b862c0aa742704b743ab9c2d
------------------------------------------------------------
Index: 2 | File: a2.txt
Content: $10000 transaction done from A TO B..
Hash: 0000f34d50a1bb631782f21e8a7230ace82b9543793bffcdb17f47f7fabbf8a1
------------------------------------------------------------
Index: 3 | File: a3.txt
Content: second
Hash: 00004b6729076ba594317ebdd85752382f76c376820c27ce412e74c7c1b319bf
------------------------------------------------------------
✅ Link [Genesis -> a1.txt] is VALID.
✅ Link [a1.txt -> a2.txt] is VALID.
✅ Link [a2.txt -> a3.txt] is VALID.

🏆 FINAL RESULT: Blockchain is fully synchronized and secure.
Chain valid

In [88]:
target_text = "$10000 transaction done from A TO B.."
new_text = "$100000000 transaction done from A TO B.."
index = -1

print(f"Searching for transaction '{target_text}'...")

for block in bc.chain:
    content = block.data.get("content", "")

    if target_text in content:
        print(f" FOUND! Target text is in Block Index: {block.index}")
       
        print(f"   Old Content: {content}")
        block.data["content"] = content.replace(target_text, new_text)
        print(f"   New Content: {block.data['content']}")
        block.nonce = 0 
        bc.mine_block(block)
        print(f"   New Hash: {block.hash}")
        
        index = block.index
        break

 
if(index == -1):
    print("target data not in blockchain")

Searching for transaction '$10000 transaction done from A TO B..'...
 FOUND! Target text is in Block Index: 2
   Old Content: $10000 transaction done from A TO B..
   New Content: $100000000 transaction done from A TO B..
   New Hash: 0000c2619d091330c98d7c199f4a0f1f66c0b357f4860571a16b84a8edb04b26


In [89]:
bc.is_chain_valid()

✅ Link [Genesis -> a1.txt] is VALID.
✅ Link [a1.txt -> a2.txt] is VALID.
❌ LINK BROKEN: [a2.txt -> a3.txt]
   Reason: previous_hash in a3.txt does not match a2.txt's hash.


False

In [90]:
if index != -1:
    print("\n--- 🔄 Repairing broken links (Making hack undetected) ---")
    
    for i in range(index + 1, len(bc.chain)):
        curr_block = bc.chain[i]
        prev_block = bc.chain[i-1]
        
        curr_block.previous_hash = prev_block.hash
        
        curr_block.nonce = 0
        bc.mine_block(curr_block)
        
        print(f"   ✅ Fixed & Re-mined Block {curr_block.index}")
        
    print("\n🏆 SUCCESS: Data changed and chain re-synchronized.")


--- 🔄 Repairing broken links (Making hack undetected) ---
   ✅ Fixed & Re-mined Block 3

🏆 SUCCESS: Data changed and chain re-synchronized.


In [91]:
bc.is_chain_valid()

✅ Link [Genesis -> a1.txt] is VALID.
✅ Link [a1.txt -> a2.txt] is VALID.
✅ Link [a2.txt -> a3.txt] is VALID.

🏆 FINAL RESULT: Blockchain is fully synchronized and secure.


True